# Análisis de Oraciones con spaCy (Modelo en Español)

Este notebook realiza un análisis de las oraciones contenidas en `ground_truth_ner.json` utilizando el modelo grande de español de spaCy (`es_core_news_lg`).

**Nota sobre la etiqueta NUM:** El modelo estándar de spaCy para español (`es_core_news_lg`) detecta entidades como `PER` (Personas), `ORG` (Organizaciones), `LOC` (Lugares) y `MISC` (Miscelánea), pero **no incluye por defecto** números o cantidades (`NUM`, `CARDINAL`) como entidades nombradas. Sin embargo, sí los detecta gramaticalmente (POS tagging). En este notebook, hemos agregado un paso personalizado para incluir los tokens etiquetados gramaticalmente como `NUM` dentro de las entidades visualizadas.

In [5]:
# Instalación del modelo grande de español de spaCy
!python -m spacy download es_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 7.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')


In [6]:
import spacy
import json
import pandas as pd
from spacy import displacy
from spacy.tokens import Span  # Necesario para crear entidades manualmente

# Cargar el modelo en español
try:
    nlp = spacy.load("es_core_news_lg")
    print("Modelo 'es_core_news_lg' cargado correctamente.")
except OSError:
    print("El modelo no se encontró. Asegúrate de haber ejecutado la celda de instalación.")

Modelo 'es_core_news_lg' cargado correctamente.


In [7]:
# Cargar los datos del archivo JSON
file_path = 'ground_truth_ner.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} oraciones para analizar.")

Se cargaron 15 oraciones para analizar.


## Procesamiento y Extracción de Entidades

A continuación, procesamos cada oración. 

**Mejora:** Se ha añadido lógica para detectar tokens con la etiqueta gramatical `NUM` (números) y convertirlos en entidades `NUM` para que aparezcan en la visualización, ya que el modelo NER por defecto no los incluye.

In [8]:
results = []

for item in data:
    text = item['text']
    doc = nlp(text)
    
    # ---------------------------------------------------------
    # PASO ADICIONAL: Agregar números como entidades manualmente
    # ---------------------------------------------------------
    new_ents = []
    for token in doc:
        if token.pos_ == "NUM":
            # Verificar que no se solape con una entidad existente
            is_ent = False
            for ent in doc.ents:
                if token.i >= ent.start and token.i < ent.end:
                    is_ent = True
                    break
            if not is_ent:
                # Crear nueva entidad con etiqueta NUM
                new_ents.append(Span(doc, token.i, token.i+1, label="NUM"))
    
    # Actualizamos las entidades del documento (combinando existentes + nuevas)
    try:
        # Es importante ordenar las entidades por posición
        all_ents = list(doc.ents) + new_ents
        all_ents.sort(key=lambda x: x.start)
        doc.ents = all_ents
    except Exception as e:
        print(f"Error al agregar entidades NUM en '{text}': {e}")
    
    # ---------------------------------------------------------

    # Extraer entidades para el reporte
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    
    # Guardar resultados
    results.append({
        'id': item['id'],
        'text': text,
        'entities': entities,
        'doc_object': doc  # Guardamos el objeto doc para visualizaciones posteriores
    })

# Crear DataFrame para visualizar mejor
df_results = pd.DataFrame(results).drop(columns=['doc_object'])
pd.set_option('display.max_colwidth', None)
df_results

,id,text,entities
0,1,Genera una cotización para el cliente Compufacil con 5 monitores led y 3 soportes de pared,"[(Compufacil, MISC), (5, NUM), (3, NUM)]"
1,2,Prepara un presupuesto urgente con 10 teclados inalámbricos y 10 ratones ópticos para enviar a Tecnosys,"[(10, NUM), (10, NUM), (Tecnosys, PER)]"
2,3,Crea una oferta comercial para Carla Santana con 1 escritorio ejecutivo modelo XG Premium,"[(Carla Santana, PER), (1, NUM), (XG Premium, MISC)]"
3,4,Genera una proforma para AndinaCorp incluye 2 laptops core i7 y 3 impresoras multifunción,"[(AndinaCorp, MISC), (2, NUM), (3, NUM)]"
4,5,Busca la última factura del cliente Velasco y Asociados y reenvíala a su correo,"[(Velasco, PER)]"
5,6,Genera la factura del pedido de compra 852025,"[(852025, NUM)]"
6,7,Verifica si la factura FA 409516 de Hierros del Pacífico ya está pagada,"[(FA, ORG), (409516, NUM), (Hierros del Pacífico, PER)]"
7,8,¿Cuántas sillas ergonómicas de oficina tenemos en la bodega de Guayaquil?,"[(Guayaquil, LOC)]"
8,9,Registra el ingreso de 50 resmas de papel A4 del proveedor Papel Mundo,"[(50, NUM), (A4, ORG), (Papel Mundo, ORG)]"
9,10,Dame el stock actual de discos duros de 1 terabyte de la marca Duradisco,"[(1, NUM), (Duradisco, ORG)]"


## Visualización Detallada

Aquí mostramos las entidades detectadas en cada oración de forma visual. Ahora deberían aparecer los números etiquetados como `NUM`.

In [9]:
# Visualizar las entidades en las primeras 5 oraciones
for item in results[:5]:
    print(f"ID: {item['id']}")
    displacy.render(item['doc_object'], style="ent", jupyter=True)
    print("-" * 50)

ID: 1


--------------------------------------------------
ID: 2


--------------------------------------------------
ID: 3


--------------------------------------------------
ID: 4


--------------------------------------------------
ID: 5


--------------------------------------------------


## Análisis Gramatical (Opcional)
Muestra los tokens, lemas y categorías gramaticales de una oración de ejemplo.

In [ ]:
# Ejemplo de análisis gramatical con la primera oración
example_doc = results[0]['doc_object']

token_data = []
for token in example_doc:
    token_data.append([token.text, token.lemma_, token.pos_, token.dep_])

df_tokens = pd.DataFrame(token_data, columns=['Texto', 'Lema', 'POS', 'Dependencia'])
df_tokens

,Texto,Lema,POS,Dependencia
0,Genera,genero,PRON,ROOT
1,una,uno,DET,det
2,cotización,cotización,NOUN,obj
3,para,para,ADP,case
4,el,el,DET,det
5,cliente,cliente,NOUN,nmod
6,Compufacil,compufacil,ADJ,amod
7,con,con,ADP,case
8,5,5,NUM,nummod
9,monitores,monitor,NOUN,nmod


: 